<a href="https://colab.research.google.com/github/kaifahmad236/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaifahmad236/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os
import getpass
import duckdb
import pandas as pd
from pathlib import Path

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("Warehouse connected!")

Warehouse connected!


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# Rule Description

The baseline rule identifies pages that already rank reasonably well but receive relatively few clicks compared to their impressions.

The rule prioritizes pages with:

- High impressions
- Low click-through rate (CTR)
- Good average search position

These pages are likely candidates for title or meta description optimization because users see the page but are not clicking it.

### Reason Codes

- LOW_CTR_HIGH_POSITION
- HIGH_IMPRESSIONS_LOW_CLICKS
- REVIEW_METADATA

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position
FROM {TABLES['fact_daily']}
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) > 500
LIMIT 10
"""

signals = con.sql(query).df()

signals["ctr"] = (
    signals["clicks"] /
    signals["impressions"]
)

signals

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr
0,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,12277.0,51.0,27.339604,0.004154
1,client_9958f0a7ae1df715,content_c899aef92518c714,30756.0,115.0,33.712361,0.003739
2,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,6016.0,18.0,35.538566,0.002992
3,client_9958f0a7ae1df715,content_ae5e5fd6edff550f,14482.0,26.0,16.941366,0.001795
4,client_9958f0a7ae1df715,content_a64143f6e4a21ffe,411801.0,2947.0,14.952705,0.007156
5,client_9958f0a7ae1df715,content_e281674658070602,14886.0,33.0,19.384006,0.002217
6,client_9958f0a7ae1df715,content_658f53fa439c66ca,983.0,2.0,16.080016,0.002035
7,client_9958f0a7ae1df715,content_da9cd3207814ec8d,13726.0,14.0,29.195981,0.001020
8,client_9958f0a7ae1df715,content_96fe7476fada560c,23432.0,395.0,28.931211,0.016857
9,client_9958f0a7ae1df715,content_5ca1b43f9a4d0b01,5399.0,8.0,47.181382,0.001482


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

# Baseline Ranking Rule

Pages receive a higher priority score when they have:

- High impressions
- Low CTR
- Good search position

The score is computed using historical information only.

Action Label:

Optimize Title and Meta Description

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position
FROM {TABLES['fact_daily']}
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) > 500
"""

# Load data
df = con.sql(query).df()

# Feature engineering
df["ctr"] = df["clicks"] / df["impressions"]

# Baseline action score
df["score"] = (
    df["impressions"] * 0.5
    + (1 - df["ctr"]) * 1000 * 0.3
    + (30 - df["avg_position"]) * 10 * 0.2
)

# Rule outputs
df["reason_code"] = "LOW_CTR_HIGH_POSITION"
df["action"] = "Optimize Title & Meta"

# Rank pages
df = df.sort_values("score", ascending=False)

# Create output directory
Path("work/outputs").mkdir(parents=True, exist_ok=True)

# Save ranked queue
output_file = "work/outputs/baseline_action_score.csv"
df.to_csv(output_file, index=False)

print(f"CSV saved successfully with {len(df)} rows.")
print(f"Location: {output_file}")

# Display Top 20 recommendations
display(df.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CSV saved successfully with 129477 rows.
Location: work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr,score,reason_code,action
22864,client_e547b89c05043229,content_eadb33b5df496f4a,2902616.0,24747.0,2.636037,0.008526,1.451660e+06,LOW_CTR_HIGH_POSITION,Optimize Title & Meta
66725,client_73cda7b4e4f265ea,content_e241d6415ac9e534,1829204.0,5714.0,4.914556,0.003124,9.149512e+05,LOW_CTR_HIGH_POSITION,Optimize Title & Meta
66301,client_73cda7b4e4f265ea,content_cf651123f1085418,1388464.0,2704.0,7.369299,0.001947,6.945767e+05,LOW_CTR_HIGH_POSITION,Optimize Title & Meta
18074,client_23a62021009f63c4,content_e8a52cf3d5988c07,1285914.0,5495.0,11.665600,0.004273,6.432924e+05,LOW_CTR_HIGH_POSITION,Optimize Title & Meta
82177,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,1265772.0,13210.0,6.203495,0.010436,6.332305e+05,LOW_CTR_HIGH_POSITION,Optimize Title & Meta
2529,client_73cda7b4e4f265ea,content_b17c1d1cb0a346d6,1245353.0,5997.0,5.755486,0.004816,6.230235e+05,LOW_CTR_HIGH_POSITION,Optimize Title & Meta
2730,client_fef1a8f436438636,content_0717da99fbd8c274,1222089.0,10160.0,11.129908,0.008314,6.113797e+05,LOW_CTR_HIGH_POSITION,Optimize Title & Meta
69093,client_73cda7b4e4f265ea,content_471d9cabce329a66,1208380.0,2985.0,6.888409,0.002470,6.045355e+05,LOW_CTR_HIGH_POSITION,Optimize Title & Meta
87491,client_e547b89c05043229,content_545bb6cc7081ded3,1177698.0,5182.0,3.071687,0.004400,5.892015e+05,LOW_CTR_HIGH_POSITION,Optimize Title & Meta
24023,client_e547b89c05043229,content_963de14b1f58978f,1135755.0,3068.0,4.935717,0.002701,5.682268e+05,LOW_CTR_HIGH_POSITION,Optimize Title & Meta


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

# Top-20 Review

The highest ranked pages were manually reviewed.

Each recommendation includes:

- Action
- Reason Code
- Confidence
- What could make the recommendation incorrect

This review is intended as a sanity check before using the baseline.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = df.head(20).copy()

top20["confidence"] = "Medium"

top20["what_would_make_it_wrong"] = (
    "Seasonal content, newly published pages, or temporary traffic fluctuations."
)

display(
top20[
[
"content_hash_id",
"score",
"action",
"reason_code",
"confidence",
"what_would_make_it_wrong"
]
]
)

,content_hash_id,score,action,reason_code,confidence,what_would_make_it_wrong
22864,content_eadb33b5df496f4a,1.451660e+06,Optimize Title & Meta,LOW_CTR_HIGH_POSITION,Medium,"Seasonal content, newly published pages, or te..."
66725,content_e241d6415ac9e534,9.149512e+05,Optimize Title & Meta,LOW_CTR_HIGH_POSITION,Medium,"Seasonal content, newly published pages, or te..."
66301,content_cf651123f1085418,6.945767e+05,Optimize Title & Meta,LOW_CTR_HIGH_POSITION,Medium,"Seasonal content, newly published pages, or te..."
18074,content_e8a52cf3d5988c07,6.432924e+05,Optimize Title & Meta,LOW_CTR_HIGH_POSITION,Medium,"Seasonal content, newly published pages, or te..."
82177,content_e7b5dd4dff461ad2,6.332305e+05,Optimize Title & Meta,LOW_CTR_HIGH_POSITION,Medium,"Seasonal content, newly published pages, or te..."
2529,content_b17c1d1cb0a346d6,6.230235e+05,Optimize Title & Meta,LOW_CTR_HIGH_POSITION,Medium,"Seasonal content, newly published pages, or te..."
2730,content_0717da99fbd8c274,6.113797e+05,Optimize Title & Meta,LOW_CTR_HIGH_POSITION,Medium,"Seasonal content, newly published pages, or te..."
69093,content_471d9cabce329a66,6.045355e+05,Optimize Title & Meta,LOW_CTR_HIGH_POSITION,Medium,"Seasonal content, newly published pages, or te..."
87491,content_545bb6cc7081ded3,5.892015e+05,Optimize Title & Meta,LOW_CTR_HIGH_POSITION,Medium,"Seasonal content, newly published pages, or te..."
24023,content_963de14b1f58978f,5.682268e+05,Optimize Title & Meta,LOW_CTR_HIGH_POSITION,Medium,"Seasonal content, newly published pages, or te..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# Weak Picks and Leakage Check

Weak Picks

This rule may incorrectly prioritize:

- Seasonal pages
- Recently published content
- Pages with temporary traffic changes
- Pages affected by external events

Leakage Check

Only historical search performance metrics were used.

No future observations, labels, or product-generated flags were included.

No client names, URLs, or private information were used.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Leakage Checklist")

print("----------------------------")

print("✓ Uses historical impressions")

print("✓ Uses historical clicks")

print("✓ Uses historical position")

print("✓ No future windows")

print("✓ No labels")

print("✓ No product flags")

print("✓ No URLs")

print("✓ No client names")

Leakage Checklist
----------------------------
✓ Uses historical impressions
✓ Uses historical clicks
✓ Uses historical position
✓ No future windows
✓ No labels
✓ No product flags
✓ No URLs
✓ No client names


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.